# İnce ayarlı model + agentic RAG — ölçüm

`agentic_rag/` deposundaki getirme katmanını kendi ince ayarlı Qwen modelimizle birleştirip 316 soruluk temiz test setinde ölçüyor.

**Not:** `agentic_rag` varsayılan olarak Together API / GLM-5.3-Flash kullanır. Bu defter onun yerine yerel Qwen3.5-4B + LoRA adapter kullanır; Together anahtarı gerekmez.

**Çalıştırma:** Colab, GPU. Hücreler sırayla, toplam ~90 dk. Hepsi aynı oturumda çalışmalı — değişkenler bellekte taşınıyor.

## 1 kurulum

`hucreler/01_rag_olcum/1_kurulum.py`

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q langchain-core langchain-huggingface langchain-qdrant langchain-text-splitters langgraph qdrant-client sentence-transformers python-dotenv
!pip install -q rank_bm25 FlagEmbedding

## 2 hazirlik

`hucreler/01_rag_olcum/2_hazirlik.py`

In [ ]:
import os, sys, glob, json, zipfile, shutil, unicodedata, collections, torch
from pathlib import Path
from google.colab import drive

os.chdir("/content")
drive.mount("/content/drive", force_remount=False)

REPO = "/content/agentic_rag_repo"
if not os.path.isdir(f"{REPO}/agentic_rag"):
    shutil.rmtree(REPO, ignore_errors=True)
    kod = os.system("git clone --branch agentic-rag --depth 1 "
                    "https://github.com/okaybuyukdeveci/Turkish-Medical-LLM-Fine-Tuning-and-Agentic-RAG.git "
                    + REPO)
    assert kod == 0 and os.path.isdir(f"{REPO}/agentic_rag"), f"klonlama basarisiz (kod {kod})"
PROJ = f"{REPO}/agentic_rag"
DATA = f"{PROJ}/data"
sys.path.insert(0, PROJ)
os.chdir(PROJ)

def anahtar(ad):
    d = unicodedata.normalize("NFD", ad)
    return "".join(c for c in d if not unicodedata.combining(c)).casefold()

gruplar = collections.defaultdict(list)
for d in Path(DATA).iterdir():
    if d.is_dir(): gruplar[anahtar(d.name)].append(d)
silinen = 0
for ds in gruplar.values():
    if len(ds) < 2: continue
    ds.sort(key=lambda p: len(p.name))
    for d in ds[1:]:
        silinen += len(list(d.rglob("*.md"))); shutil.rmtree(d)
kalan = glob.glob(f"{DATA}/**/*.md", recursive=True)
print(f"korpus: {len(kalan)} markdown, {len(list(Path(DATA).iterdir()))} ders"
      + (f" (Unicode kopyasi silindi: {silinen})" if silinen else ""))

ZIP_PATH = "/content/drive/MyDrive/tus_data/TUS_Dataset_Final-20261002T154401Z-1-001.zip"
HEDEF = "/content/tus_veri"
if not glob.glob(f"{HEDEF}/**/test_cevap.jsonl", recursive=True):
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(HEDEF)
DATA_DIR = str(Path(glob.glob(f"{HEDEF}/**/test_cevap.jsonl", recursive=True)[0]).parent)
test_rows = [{"question": r["soru"], "letter": r["dogru_cevap"],
              "group": (r.get("meta") or {}).get("group", "?")}
             for r in (json.loads(l) for l in open(f"{DATA_DIR}/test_cevap.jsonl", encoding="utf-8"))]
print(f"test: {len(test_rows)} soru")

from scripts.settings import RagSettings
from scripts.indexing.indexer import ensure_index
from scripts.retrieval.retriever import load_retriever

settings = RagSettings(embedding_device="cuda" if torch.cuda.is_available() else "cpu")
settings.validate()
print(f"gomme: {settings.embedding_model} ({settings.embedding_device}) | k={settings.retrieval_k}")

print("\nindeks kuruluyor (5-15 dk)")
rapor = ensure_index(settings, force_recreate=silinen > 0)
print("indeks:", rapor)

retriever = load_retriever(settings)
ornek = retriever.invoke("akut miyokard infarktusunde en erken yukselen belirtec")
print(f"\nretrieval denemesi: {len(ornek)} belge")
print("ilk belge:", ornek[0].page_content[:300].replace("\n", " "))

## 3 model v2

`hucreler/01_rag_olcum/3_model_v2.py`

In [ ]:
import re, torch
from pathlib import Path
from unsloth import FastLanguageModel
from peft import PeftModel

MODEL_NAME  = "Qwen/Qwen3.5-4B"
ADAPTER_V2  = "/content/drive/MyDrive/tus_sft/qwen35_4b_v2/final_adapter"
ADAPTER_V1  = "/content/drive/MyDrive/tus_sft/qwen35_4b/final_adapter"
MAX_SEQ_LEN, SEED = 4096, 42

assert Path(ADAPTER_V2).is_dir(), f"v2 adapter yok: {ADAPTER_V2}"

SYSTEM_SFT = ("Sen bir tıp uzmanısın. Aşağıdaki TUS sorusunu çöz. "
              "İlk satıra 'Doğru cevap: <harf>) <şık metni>' yaz, ardından kısa bir açıklama ver.")

SYSTEM_RAG = ("Sen bir tıp uzmanısın. Aşağıda bir TUS sorusu ve ders kitaplarından alınmış "
              "ilgili bölümler var. Kaynak metni kullanarak soruyu çöz. "
              "İlk satıra 'Doğru cevap: <harf>) <şık metni>' yaz, ardından kısa bir açıklama ver. "
              "Kaynak metin soruyu cevaplamıyorsa kendi tıbbi bilgini kullan.")

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME, max_seq_length=MAX_SEQ_LEN, dtype=torch.bfloat16, load_in_4bit=False)
tok = getattr(tokenizer, "tokenizer", tokenizer)
if tok.pad_token is None: tok.pad_token = tok.eos_token

model = PeftModel.from_pretrained(model, ADAPTER_V2, adapter_name="v2")
AKTIF = "v2"

V1_VAR = Path(ADAPTER_V1).is_dir()
if V1_VAR:
    try:
        model.load_adapter(ADAPTER_V1, adapter_name="v1")
        print("v1 adapter da yuklendi — esli karsilastirma yapilabilir")
    except Exception as e:
        V1_VAR = False
        print("v1 adapter yuklenemedi, sadece v2 olculecek:", type(e).__name__, e)
else:
    print("v1 adapter Drive'da bulunamadi, sadece v2 olculecek")

def adapter_sec(ad):
    global AKTIF
    model.set_adapter(ad); AKTIF = ad
    print(f"aktif adapter: {ad}")

FastLanguageModel.for_inference(model)
model.set_adapter("v2")
print("model hazir | aktif adapter: v2")

def prompt(sistem, kullanici):
    msgs = [{"role": "system", "content": sistem}, {"role": "user", "content": kullanici}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

SON = re.compile(r"Doğru cevap\s*:?\s*\(?([A-E])\)?")
HER = re.compile(r"(?:Doğru\s*)?(?:cevap|yanıt)\s*[:\-]?\s*\(?([A-E])\)?", re.I)
def parse_cevap(t):
    c = re.sub(r"<think>.*?</think>", "", t or "", flags=re.S).strip()
    m = SON.search(c) or HER.search(c) or re.search(r"\b([A-E])\)", c)
    return m.group(1).upper() if m else None

@torch.no_grad()
def uret(prompts, max_new_tokens=96, bs=8, tekrar_cezasi=1.0, etiket=""):
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(prompts), bs):
        b = prompts[i:i+bs]
        enc = tok(b, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        g = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                           repetition_penalty=tekrar_cezasi, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"  {etiket}{min(i+bs,len(prompts))}/{len(prompts)}", end="\r")
    print()
    return outs

def dogruluk(outs, rows, etiket=""):
    p = [parse_cevap(o) for o in outs]
    acc = sum(a == r["letter"] for a, r in zip(p, rows)) / len(rows)
    yok = sum(a is None for a in p)
    print(f"{etiket}dogruluk {acc:.1%} | cevapsiz {yok}/{len(rows)}")
    return acc, p

MAX_BAGLAM = 6000
def baglam_kur(docs, limit=MAX_BAGLAM):
    parcalar, toplam = [], 0
    for i, d in enumerate(docs, 1):
        t = d.page_content.strip()
        kaynak = (d.metadata or {}).get("source", "?")
        blok = f"[Kaynak {i} — {Path(kaynak).stem}]\n{t}"
        if toplam + len(blok) > limit:
            kalan = limit - toplam
            if kalan > 400: parcalar.append(blok[:kalan])
            break
        parcalar.append(blok); toplam += len(blok)
    return "\n\n".join(parcalar)

print("\n[referans] RAG'siz olcum — v2")
sade_outs = uret([prompt(SYSTEM_SFT, r["question"]) for r in test_rows], 64, bs=16, etiket="[sade] ")
sade_acc, sade_pred = dogruluk(sade_outs, test_rows, "[sade v2] ")
print("beklenen ~71,8% (SFT v2 olcumu) | v1: 70,9%")

## 4 rag duz

`hucreler/01_rag_olcum/4_rag_duz.py`

In [ ]:
print("[duz RAG] belgeler getiriliyor (k=6, baglam 6000)")
B_docs = []
for i, r in enumerate(test_rows, 1):
    B_docs.append(retriever.invoke(r["question"]))
    print(f"  getirme {i}/{len(test_rows)}", end="\r")
print(f"\nortalama belge: {sum(len(d) for d in B_docs)/len(B_docs):.1f}")

B_pr = [prompt(SYSTEM_RAG, f"KAYNAK METİN:\n{baglam_kur(d)}\n\n---\n\nSORU:\n{r['question']}")
        for r, d in zip(test_rows, B_docs)]
uz = sorted(len(tok(p)["input_ids"]) for p in B_pr)
print(f"prompt token p50 {uz[len(uz)//2]} | p95 {uz[int(len(uz)*.95)]} | max {uz[-1]}")

B_outs = uret(B_pr, 64, bs=4, etiket="[duz RAG] ")
B_acc, B_pred = dogruluk(B_outs, test_rows, "[duz RAG v2] ")
print(f"v1 ayni ayarda: 83,9%  |  v2: {B_acc:.1%}")

## 5 hibrit rerank

`hucreler/01_rag_olcum/5_hibrit_rerank.py`

In [ ]:
import re, unicodedata, torch, gc
from rank_bm25 import BM25Okapi
from FlagEmbedding import FlagReranker
from scripts.indexing.loader import load_markdown_documents
from scripts.indexing.chunker import ChunkingConfig, chunk_parent_child_documents

LIMIT, ADAY_N, TUT_N = 6000, 30, 6

print("parent chunk'lar uretiliyor (Qdrant'a dokunulmuyor)")
belgeler = load_markdown_documents(DATA)
parents, _ = chunk_parent_child_documents(
    belgeler,
    parent_config=ChunkingConfig(max_tokens=settings.parent_chunk_size_tokens, overlap_tokens=0),
    child_config=ChunkingConfig(max_tokens=settings.chunk_size_tokens,
                                overlap_tokens=settings.chunk_overlap_tokens),
)
print(f"parent: {len(parents)}")

def jeton(s):
    s = unicodedata.normalize("NFD", s.lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    return [w for w in re.findall(r"[a-z0-9]+", s) if len(w) > 2]

print("BM25 kuruluyor")
bm25 = BM25Okapi([jeton(p.page_content) for p in parents])
icerik_idx = {p.page_content: i for i, p in enumerate(parents)}

def rrf(siralar, k=60):
    puan = {}
    for sira in siralar:
        for yer, kimlik in enumerate(sira):
            puan[kimlik] = puan.get(kimlik, 0.0) + 1.0 / (k + yer + 1)
    return [kimlik for kimlik, _ in sorted(puan.items(), key=lambda x: -x[1])]

print("reranker yukleniyor")
reranker = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=True, devices=["cuda:0"])

print(f"\nhibrit aday havuzu ({ADAY_N}) + yeniden siralama")
eski_k = retriever.child_retriever.search_kwargs.get("k", settings.retrieval_k)
retriever.child_retriever.search_kwargs["k"] = ADAY_N
HR_docs = []
for i, r in enumerate(test_rows, 1):
    yogun = retriever.invoke(r["question"])
    yogun_idx = [x for x in (icerik_idx.get(d.page_content) for d in yogun) if x is not None]
    seyrek_idx = list(bm25.get_top_n(jeton(r["question"]), range(len(parents)), n=ADAY_N))
    aday = [parents[j] for j in rrf([yogun_idx, seyrek_idx])[:ADAY_N]]
    if len(aday) <= TUT_N:
        HR_docs.append(aday)
    else:
        skor = reranker.compute_score([[r["question"], d.page_content[:2000]] for d in aday],
                                      normalize=True)
        sirali = sorted(zip(skor, range(len(aday))), reverse=True)
        HR_docs.append([aday[j] for _, j in sirali[:TUT_N]])
    print(f"  {i}/{len(test_rows)}", end="\r")
retriever.child_retriever.search_kwargs["k"] = eski_k
print()

del reranker; gc.collect(); torch.cuda.empty_cache()

HR_pr = [prompt(SYSTEM_RAG, f"KAYNAK METİN:\n{baglam_kur(d, LIMIT)}\n\n---\n\nSORU:\n{r['question']}")
         for r, d in zip(test_rows, HR_docs)]

adapter_sec("v2")
HR_outs = uret(HR_pr, 64, bs=4, etiket="[hib+rr v2] ")
HR_acc, HR_pred = dogruluk(HR_outs, test_rows, "[hib+rr v2] ")

HR_v1_acc, HR_v1_pred = None, None
if V1_VAR:
    adapter_sec("v1")
    HR_v1_outs = uret(HR_pr, 64, bs=4, etiket="[hib+rr v1] ")
    HR_v1_acc, HR_v1_pred = dogruluk(HR_v1_outs, test_rows, "[hib+rr v1] ")
    adapter_sec("v2")
    print(f"\nayni belgeler, ayni prompt — v1 {HR_v1_acc:.1%} vs v2 {HR_acc:.1%}")
else:
    print(f"\nv1 olculemedi; gecen seferki v1 sayisi 88,9% (ayri oturum, esli test yapilamaz)")

## 6 ozet

`hucreler/01_rag_olcum/6_ozet.py`

In [ ]:
import os, json
from math import comb

def mcnemar(p1, p2):
    a = sum(x != r["letter"] and y == r["letter"] for x, y, r in zip(p1, p2, test_rows))
    b = sum(x == r["letter"] and y != r["letter"] for x, y, r in zip(p1, p2, test_rows))
    n = a + b
    p = (sum(comb(n, k) for k in range(max(a, b), n + 1)) / 2**n * 2) if n else 1.0
    return a, b, min(p, 1.0)

print("="*66)
print(f"{'kurulum':<34}{'v1':>10}{'v2':>10}")
print("-"*66)
print(f"{'RAG yok':<34}{'70,9%':>10}{sade_acc:>10.1%}")
print(f"{'duz RAG (k=6, 6000)':<34}{'83,9%':>10}{B_acc:>10.1%}")
v1_hr = f"{HR_v1_acc:.1%}" if HR_v1_acc is not None else "88,9%*"
print(f"{'hibrit + rerank':<34}{v1_hr:>10}{HR_acc:>10.1%}")
print("="*66)
if HR_v1_acc is None:
    print("* ayri oturumdan, esli test yapilamaz")

print("\nv2 icinde (esli, McNemar):")
for ad, acc, pred in [("duz RAG", B_acc, B_pred), ("hibrit + rerank", HR_acc, HR_pred)]:
    a, b, p = mcnemar(sade_pred, pred)
    print(f"  RAG yok -> {ad:<18} {acc-sade_acc:+.1%}  lehte {a} aleyhte {b}  p={p:.4f}"
          + ("  ANLAMLI" if p < 0.05 else ""))
a, b, p = mcnemar(B_pred, HR_pred)
print(f"  duz RAG -> hibrit+rerank    {HR_acc-B_acc:+.1%}  lehte {a} aleyhte {b}  p={p:.4f}"
      + ("  ANLAMLI" if p < 0.05 else ""))

if HR_v1_pred is not None:
    a, b, p = mcnemar(HR_v1_pred, HR_pred)
    print(f"\nadapter karsilastirmasi (ayni belgeler, hibrit+rerank):")
    print(f"  v1 -> v2  {HR_acc-HR_v1_acc:+.1%}  lehte {a} aleyhte {b}  p={p:.4f}"
          + ("  ANLAMLI" if p < 0.05 else "  anlamli degil"))

OUT = "/content/drive/MyDrive/tus_sft/rag_sonuc"
os.makedirs(OUT, exist_ok=True)
kayit = {
    "adapter": "qwen35_4b_v2",
    "korpus": "temizlik oncesi (repo hali)",
    "sade": sade_acc, "duz_rag": B_acc, "hibrit_rerank": HR_acc,
    "hibrit_rerank_v1": HR_v1_acc,
    "tahminler": {"sade": sade_pred, "duz_rag": B_pred,
                  "hibrit_rerank": HR_pred, "hibrit_rerank_v1": HR_v1_pred},
    "dogru": [r["letter"] for r in test_rows],
}
with open(f"{OUT}/v2_korpus_oncesi.json", "w", encoding="utf-8") as f:
    json.dump(kayit, f, ensure_ascii=False, indent=2)
print("\nkaydedildi:", f"{OUT}/v2_korpus_oncesi.json")
print("tahminler de kaydedildi — korpus temizligi sonrasi esli test bununla yapilacak")

## 7 teshis

`hucreler/01_rag_olcum/7_teshis.py`

In [ ]:
import re, unicodedata

def sadelestir(s):
    s = unicodedata.normalize("NFD", (s or "").lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9 ]+", " ", s)

def sik_metni(soru, harf):
    m = re.search(rf"^\s*{harf}\s*[\)\.]\s*(.+?)\s*$", soru, re.M)
    return m.group(1).strip() if m else None

def ortusme(metin, baglam):
    a, b = sadelestir(metin).split(), set(sadelestir(baglam).split())
    a = [w for w in a if len(w) > 3]
    return sum(w in b for w in a) / len(a) if a else 0.0

dogru   = [r["letter"] for r in test_rows]
gerile  = [i for i in range(len(test_rows)) if HR_v1_pred[i] == dogru[i] and HR_pred[i] != dogru[i]]
ilerle  = [i for i in range(len(test_rows)) if HR_v1_pred[i] != dogru[i] and HR_pred[i] == dogru[i]]
print(f"v2'nin kaybettigi: {len(gerile)} | kazandigi: {len(ilerle)}")

print("\n[1] v2 baglami mi yok sayiyor? (RAG'li cevap == RAG'siz cevabi mi)")
for ad, idx in [("gerileyen", gerile), ("tum sorular", range(len(test_rows)))]:
    idx = list(idx)
    ayni_v2 = sum(HR_pred[i] == sade_pred[i] for i in idx) / len(idx)
    print(f"  {ad:<14} v2 RAG cevabi = v2 RAG'siz cevabi: %{100*ayni_v2:.0f}  (n={len(idx)})")

print("\n[2] dogru cevabin kaniti baglamda var miydi?")
for ad, idx in [("gerileyen", gerile), ("tum sorular", range(len(test_rows)))]:
    idx = list(idx); skor = []
    for i in idx:
        mt = sik_metni(test_rows[i]["question"], dogru[i])
        if mt: skor.append(ortusme(mt, baglam_kur(HR_docs[i], 6000)))
    if skor:
        print(f"  {ad:<14} dogru sik metninin baglamla ortusmesi: ort %{100*sum(skor)/len(skor):.0f}  (n={len(skor)})")

print("\n[3] cikti uzunlugu (karakter)")
for ad, outs in [("v1", HR_v1_outs), ("v2", HR_outs)]:
    uz = sorted(len(o) for o in outs)
    print(f"  {ad}: p50 {uz[len(uz)//2]} | p95 {uz[int(len(uz)*.95)]} | max {uz[-1]}")
    kesik = sum(not re.search(r"[.!?]\s*$", o.strip()) for o in outs)
    print(f"      cumle ortasinda kesilen: {kesik}/{len(outs)}")

print("\n[4] gerileyen sorulardan ilk 5 — yan yana")
for i in gerile[:5]:
    print("\n" + "="*70)
    print(f"soru {i} | dogru {dogru[i]} | v1 {HR_v1_pred[i]} | v2 {HR_pred[i]} | v2 RAG'siz {sade_pred[i]}")
    print("-"*70)
    print("SORU :", test_rows[i]["question"][:230].replace("\n", " "))
    print("v1   :", HR_v1_outs[i][:220].replace("\n", " "))
    print("v2   :", HR_outs[i][:220].replace("\n", " "))

## 8 gecikme

`hucreler/01_rag_olcum/8_gecikme.py`

In [ ]:
import time, statistics, torch
from FlagEmbedding import FlagReranker

ADAY_N, TUT_N, LIMIT = 30, 6, 6000

print("reranker yukleniyor (urunde surekli bellekte durur, olcume dahil degil)")
reranker = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=True, devices=["cuda:0"])
adapter_sec("v1")

def tam_akis(soru, max_new_tokens=64):
    z = {}
    t0 = time.perf_counter()
    retriever.child_retriever.search_kwargs["k"] = ADAY_N
    yogun = [x for x in (icerik_idx.get(d.page_content) for d in retriever.invoke(soru)) if x is not None]
    t1 = time.perf_counter(); z["yogun"] = t1 - t0

    seyrek = list(bm25.get_top_n(jeton(soru), range(len(parents)), n=ADAY_N))
    t2 = time.perf_counter(); z["bm25"] = t2 - t1

    aday = [parents[j] for j in rrf([yogun, seyrek])[:ADAY_N]]
    skor = reranker.compute_score([[soru, d.page_content[:2000]] for d in aday], normalize=True)
    secilen = [aday[j] for _, j in sorted(zip(skor, range(len(aday))), reverse=True)[:TUT_N]]
    t3 = time.perf_counter(); z["rerank"] = t3 - t2

    p = prompt(SYSTEM_RAG, f"KAYNAK METİN:\n{baglam_kur(secilen, LIMIT)}\n\n---\n\nSORU:\n{soru}")
    enc = tok(p, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        g = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tok.pad_token_id)
    t4 = time.perf_counter(); z["uretim"] = t4 - t3
    z["toplam"] = t4 - t0
    z["token"] = int(g[0].shape[0] - enc["input_ids"].shape[1])
    z["metin"] = tok.decode(g[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return z

sorular = [r["question"] for r in test_rows[:12]]
tam_akis(sorular[0])

for etiket, mnt in [("sadece cevap (64 token)", 64), ("tam aciklama (300 token)", 300)]:
    olcum = [tam_akis(s, mnt) for s in sorular]
    print(f"\n{etiket} — {len(sorular)} soru, tek tek (toplu islem yok)")
    for ad in ("yogun", "bm25", "rerank", "uretim"):
        d = [o[ad] for o in olcum]
        print(f"  {ad:<9} medyan {statistics.median(d):>6.2f} sn   (%{100*statistics.median(d)/statistics.median([o['toplam'] for o in olcum]):.0f})")
    top = [o["toplam"] for o in olcum]
    tk  = [o["token"] for o in olcum]
    print(f"  {'TOPLAM':<9} medyan {statistics.median(top):>6.2f} sn | p95 {sorted(top)[int(len(top)*.95)]:.2f} | en yavas {max(top):.2f}")
    print(f"  uretim hizi: {statistics.median(tk)/statistics.median([o['uretim'] for o in olcum]):.0f} token/sn")

print("\nornek cikti (300 token):")
print(tam_akis(sorular[1], 300)["metin"][:500])
print("\nNot: A100-40GB, model ve reranker bellekte. Soguk baslatma (model yukleme ~2 dk) dahil degil.")